# Notebook 02 — Baseline Classifier

## Goal
Train three classifiers (Random Forest + SMOTE, AdaBoost + SMOTE, KNN baseline)
on the `prepared_wells.csv` feature set and evaluate them with:
- Precision / Recall / F1
- ROC-AUC and Average Precision
- **Cost-Benefit ROC curve** — the key business metric for the resource-allocation framing

## Feature Framing (No Data Leakage)
Features are restricted to those realistically known **before** a well is chemically tested:
- `latitude`, `longitude` — geospatial position
- `groundwater_level_m` — depth to water table (quarterly manual readings)
- `state_encoded`, `district_encoded` — regional categorical context

Target: `is_contaminated` (1 if EC > 3000 µS/cm, else 0).

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

from src.data_prep import run_data_prep
from src.classifier import run_classifier_pipeline, cost_benefit_analysis

PROCESSED_DIR = '../data/processed'
FIGURES_DIR = '../figures'

## Step 1 — Load & Prepare Data

In [ ]:
X_train, X_test, y_train, y_test, encoders, df_prepared = run_data_prep(PROCESSED_DIR)

print(f'Training set : {len(X_train):,} samples  |  Contamination rate: {y_train.mean():.2%}')
print(f'Test set     : {len(X_test):,} samples  |  Contamination rate: {y_test.mean():.2%}')
X_train.head()

## Step 2 — Train All Models

In [ ]:
results = run_classifier_pipeline(X_train, X_test, y_train, y_test, figures_dir=FIGURES_DIR)

## Step 3 — Detailed Classification Reports

In [ ]:
for name, res in results.items():
    print(f'\n{'='*60}')
    print(f'  {name}')
    print(f'  ROC-AUC: {res["roc_auc"]:.4f}  |  Avg Precision: {res["avg_precision"]:.4f}')
    print(f'{'='*60}')
    print(res['report'])

## Step 4 — View Saved Figures

All plots have been saved to `../figures/`. The key figures are:
- `roc_curve.png` — Comparative ROC curves
- `pr_curve.png` — Precision-Recall curves (most informative for imbalanced data)
- `cost_benefit_curve.png` — Total operational cost vs. decision threshold
- `confusion_matrix_random_forest_+_smote.png` — At the cost-optimal threshold

### Interpretation Note
The cost-benefit curve is the **most important plot** for the viva.
It directly answers: *"At what threshold should we flag a well for testing,
given that missing a contaminated well costs 10× more than a wasted test?"*

In [ ]:
from IPython.display import Image, display

for fname in ['roc_curve.png', 'pr_curve.png', 'cost_benefit_curve.png']:
    fpath = os.path.join(FIGURES_DIR, fname)
    if os.path.exists(fpath):
        print(f'\n--- {fname} ---')
        display(Image(fpath))